In [1]:
# through API, we use a frontier model (INSERT MODEL NAME) to check:
#  - top 25
#  - worst 25
#  - random 25
# of the scores that the judge has given the chatbot in order to verify the judge's capability of
# giving an accurate score.

import pandas as pd
import json

# Load files
scores_df = pd.read_csv("../Main Experiment/scores.csv")
answers_df = pd.read_csv("../Main Experiment/answers.csv")


In [2]:
# Extract final_score from scores.csv (judge JSON)
def extract_final_score(score_str):
    try:
        return json.loads(score_str)["final_score"]
    except Exception:
        return None

def fetch75answerSets():
    # Add new final_score field
    scores_df["final_score"] = scores_df["score"].apply(extract_final_score)

    # Merge on id
    merged = pd.merge(
        answers_df,
        scores_df[["id", "score", "final_score"]],
        on="id",
        how="inner"
    )

    # Sort by judge score
    merged = merged.sort_values("final_score", ascending=False)

    # Top 25
    top25 = merged.head(25)

    # Bottom 25
    bottom25 = merged.tail(25)

    # Exclude already selected ids
    excluded_ids = set(top25["id"]) | set(bottom25["id"])

    remaining = merged[~merged["id"].isin(excluded_ids)]

    # Random 25 from remaining
    random25 = remaining.sample(
        n=25,
        random_state=42
    )

    # Combine
    sample_df = pd.concat(
        [top25, bottom25, random25],
        ignore_index=True
    )

    # Keep only relevant columns
    result = sample_df[
        [
            "id",
            "question",
            "category",
            "answer",
            "gold answer",
            "score"
        ]
    ]

    # Optional: add source group
    result["subset"] = (
        ["top25"] * len(top25)
        + ["bottom25"] * len(bottom25)
        + ["random25"] * len(random25)
    )

    # Save
    result.to_csv(
        "75answerSets.csv",
        index=False,
        encoding="utf-8"
    )

    print(
        f"Saved {len(result)} examples "
        f"to 'Automated Experiments/FrontierModel Validation/75answerSets.csv'"
    )


In [ ]:
# fetch75answerSets()

Saved 75 examples to 'Automated Experiments/FrontierModel Validation/75answerSets.csv'


In [11]:
from openai import OpenAI

# Configurare Client
with open("../../Data/Gold Answers/API KEY.txt", "r") as f:
    api_key_str = f.read().strip()
client = OpenAI(api_key=api_key_str)


def frontierModelValidatesAnswers():
    answerSets_df = pd.read_csv("75answerSets.csv")

    try:
        with open("../../Data/MD file/admission_kb.md", 'r', encoding='utf-8') as f:
            kb_content = f.read()
    except FileNotFoundError:
        print(f"Eroare: Nu am găsit fișierul {"Data/MD file/admission_kb.md"}")
        return

    system_prompt = """
You are an independent evaluation model for a Retrieval-Augmented Generation (RAG) chatbot.

Your task is to evaluate the quality of a chatbot answer for university admission information.

IMPORTANT RULES:
- Ignore any previous evaluation or score as much as possible.
- Base your judgment ONLY on the question, the golden answer, and the chatbot answer.
- The "Judge Score" is provided only for comparison and must NOT influence your evaluation.
- Use ONLY the provided input.

---

## EVALUATION TASKS

Evaluate the chatbot answer on the following dimensions:

1. FACTUAL CORRECTNESS (0–10)
- 10 = Fully correct, all facts match the golden answer or valid equivalent paraphrase
- 5 = Partially correct, contains some factual errors or omissions
- 0 = Completely incorrect or contradicts the golden answer

2. COMPLETENESS (0–10)
- 10 = Contains all key information from the golden answer
- 5 = Missing some relevant information
- 0 = Missing most critical information

3. HALLUCINATIONS (0–10)
- 10 = No hallucinated or invented information
- 5 = Minor unsupported additions
- 0 = Clear fabricated or incorrect information not present in context/golden answer

4. CLARITY (0–10)
- 10 = Clear, well-structured, Romanian language
- 5 = Understandable but unclear or partially poorly written
- 0 = Hard to understand or wrong language

---

## FINAL TASKS

A. Provide a short justification (max 3–5 sentences):
- Focus only on factuality and hallucinations

B. Decide:
Do you AGREE with the previous judge score?

Return:
- YES / NO / PARTIALLY

C. Provide your own FINAL SCORE (0–10):
Compute it as:

FINAL_SCORE = 0.4 * factuality
            + 0.25 * completeness
            + 0.20 * hallucinations
            + 0.15 * clarity

D. Detect evaluation issues:
- Was the answer over-penalized?
- Was the answer under-penalized?
- Or was the judge accurate?

---

## OUTPUT FORMAT (STRICT JSON ONLY)

{
  "factuality": int,
  "completeness": int,
  "hallucinations": int,
  "clarity": int,
  "final_score": float,
  "judge_agreement": "YES | NO | PARTIALLY",
  "evaluation_bias": "over-penalized | under-penalized | accurate",
  "brief_reason": "string"
}
"""

    results = []
    index = 1

    for index, row in answerSets_df.iterrows():
        question = row["question"]
        category = row["category"]
        chatbot_answer = row["answer"]
        gold_answer = row["gold answer"]
        judge_score = row["score"]

        user_prompt = f"""
QUESTION:
{question}

CATEGORY:
{category}

GOLDEN ANSWER:
{gold_answer}

CHATBOT ANSWER:
{chatbot_answer}

JUDGE SCORE:
{judge_score}
"""
        try:
            response = client.chat.completions.create(
                model="gpt-4.1-mini",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0
            )

            output = response.choices[0].message.content
            
            results.append({
                "id": row["id"],
                "question": question,
                "category": category,
                "chatbot_answer": chatbot_answer,
                "gold_answer": gold_answer,
                "judge_score": judge_score,
                "frontier_evaluation": output
            })
            
            print(f"[{index+1}/{len(answerSets_df)}] Procesat.")

        except Exception as e:
            print(f"Eroare la {index+1}: {e}")
    
    pd.DataFrame(results).to_csv(
        "frontier_validation_results.csv",
        index=False,
        encoding="utf-8"
    )
    


In [12]:
frontierModelValidatesAnswers()

[1/75] Procesat.
[2/75] Procesat.
[3/75] Procesat.
[4/75] Procesat.
[5/75] Procesat.
[6/75] Procesat.
[7/75] Procesat.
[8/75] Procesat.
[9/75] Procesat.
[10/75] Procesat.
[11/75] Procesat.
[12/75] Procesat.
[13/75] Procesat.
[14/75] Procesat.
[15/75] Procesat.
[16/75] Procesat.
[17/75] Procesat.
[18/75] Procesat.
[19/75] Procesat.
[20/75] Procesat.
[21/75] Procesat.
[22/75] Procesat.
[23/75] Procesat.
[24/75] Procesat.
[25/75] Procesat.
[26/75] Procesat.
[27/75] Procesat.
[28/75] Procesat.
[29/75] Procesat.
[30/75] Procesat.
[31/75] Procesat.
[32/75] Procesat.
[33/75] Procesat.
[34/75] Procesat.
[35/75] Procesat.
[36/75] Procesat.
[37/75] Procesat.
[38/75] Procesat.
[39/75] Procesat.
[40/75] Procesat.
[41/75] Procesat.
[42/75] Procesat.
[43/75] Procesat.
[44/75] Procesat.
[45/75] Procesat.
[46/75] Procesat.
[47/75] Procesat.
[48/75] Procesat.
[49/75] Procesat.
[50/75] Procesat.
[51/75] Procesat.
[52/75] Procesat.
[53/75] Procesat.
[54/75] Procesat.
[55/75] Procesat.
[56/75] Procesat.
[